# XG Boost using real (unsampled) data for training
## The same process was used for SMOTE-balanced data and SMOTE-balanced synthetic data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score
)
from sklearn.metrics import confusion_matrix

In [ ]:
real_ = pd.read_csv(r'..../Train/SGA_train.csv')
real_df = real_.drop(columns=['A', 'B','C', 'D'])

In [ ]:
unsampled_test_ = pd.read_csv(r'..../Test/SGA_test.csv')
unsampled_test_ = unsampled_test_.drop(columns=['A', 'B','C', 'D'])


# Split and drop label

In [ ]:
# Assuming 'df' is your DataFrame
X_r = real_df.drop('Necrotizing Enterocolitis ', axis=1) # Features
y_r = real_df['Necrotizing Enterocolitis '] # Target variable

In [ ]:
# Assuming 'df' is your DataFrame
X_test_r = unsampled_test_.drop('Necrotizing Enterocolitis ', axis=1) # Features
y_test_r = unsampled_test_['Necrotizing Enterocolitis '] # Target variable

In [ ]:
# ---------------------------------------------------
# 1. Compute scale_pos_weight from TRAIN labels ONLY
# ---------------------------------------------------
neg = np.sum(y_r == 0)
pos = np.sum(y_r == 1)

scale_pos_weight = neg / pos

print("Number of controls:", neg)
print("Number of NEC cases:", pos)
print("scale_pos_weight:", scale_pos_weight)

# XGBoost 

In [ ]:
 from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    max_delta_step=1,
    random_state=42,
    n_jobs=-1
)

xgb_clf.fit(X_r, y_r)

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
    precision_recall_curve,
    confusion_matrix
)

from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

In [ ]:
y_proba_test = xgb_clf.predict_proba(X_test_r)[:, 1]
y_pred_test = (y_proba_test >= 0.5).astype(int)

In [ ]:
test_point_metrics = pd.DataFrame({
    "Metric": [
        "AUROC",
        "AUPRC",
        "Recall",
        "Precision",
        "F1",
        "MCC",
        "Brier Score"
    ],

    "Value": [
        roc_auc_score(y_test_r, y_proba_test),
        average_precision_score(y_test_r, y_proba_test),
        recall_score(y_test_r, y_pred_test, zero_division=0),
        precision_score(y_test_r, y_pred_test, zero_division=0),
        f1_score(y_test_r, y_pred_test, zero_division=0),
        matthews_corrcoef(y_test_r, y_pred_test),
        brier_score_loss(y_test_r, y_proba_test)
    ]
})

print("XGBoost - Real Train → Real Test:")
display(test_point_metrics)

# Bootstrap real

In [ ]:
def bootstrap_ci_binary(
    y_true,
    y_proba,
    y_pred,
    n_boot=1000,
    random_state=42
):

    rng = np.random.RandomState(random_state)
    n = len(y_true)
    indices = np.arange(n)

    boot_auroc = []
    boot_auprc = []
    boot_rec = []
    boot_prec = []
    boot_f1 = []
    boot_mcc = []
    boot_brier = []

    for _ in range(n_boot):

        sample_idx = rng.choice(
            indices,
            size=n,
            replace=True
        )

        if hasattr(y_true, "iloc"):
            y_b = y_true.iloc[sample_idx]
        else:
            y_b = y_true[sample_idx]

        proba_b = y_proba[sample_idx]
        pred_b = y_pred[sample_idx]

        # AUROC requires both classes
        if len(np.unique(y_b)) == 2:
            boot_auroc.append(
                roc_auc_score(y_b, proba_b)
            )

        # AUPRC requires at least one positive
        if np.sum(y_b == 1) > 0:
            boot_auprc.append(
                average_precision_score(y_b, proba_b)
            )

        boot_rec.append(
            recall_score(
                y_b,
                pred_b,
                zero_division=0
            )
        )

        boot_prec.append(
            precision_score(
                y_b,
                pred_b,
                zero_division=0
            )
        )

        boot_f1.append(
            f1_score(
                y_b,
                pred_b,
                zero_division=0
            )
        )

        boot_mcc.append(
            matthews_corrcoef(
                y_b,
                pred_b
            )
        )

        boot_brier.append(
            brier_score_loss(
                y_b,
                proba_b
            )
        )

    results = {
        "AUROC": np.array(boot_auroc),
        "AUPRC": np.array(boot_auprc),
        "Recall": np.array(boot_rec),
        "Precision": np.array(boot_prec),
        "F1": np.array(boot_f1),
        "MCC": np.array(boot_mcc),
        "Brier Score": np.array(boot_brier)
    }

    rows = []

    for metric, values in results.items():

        rows.append({
            "Metric": metric,
            "Mean": np.mean(values),
            "CI_lower_95": np.percentile(values, 2.5),
            "CI_upper_95": np.percentile(values, 97.5)
        })

    return pd.DataFrame(rows)

In [ ]:
test_ci_xgb = bootstrap_ci_binary(
    y_test_r,
    y_proba_test,
    y_pred_test,
    n_boot=1000,
    random_state=42
)

display(test_ci_xgb)

# Save probabilities

In [ ]:
np.save("XGB_real_proba.npy", y_proba_test)

### Similarly, probabilities were saved as XGB_smote_proba, XGB_synthetic_proba, and XGB_smote_synthetic_proba

#### These probablilties were used in file Model_Results